<a href="https://colab.research.google.com/github/OmniGrid7/Movie-Review-Sentiment-Analysis/blob/main/Movie_review_setiment_analysis_using_IMDb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Setup

In [ ]:
!pip install -q transformers datasets evaluate accelerate lime wordcloud


In [ ]:
import os
import re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset

from datasets import load_dataset
import evaluate

from transformers import (
    BertTokenizerFast,
    BertForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    roc_auc_score,
    ConfusionMatrixDisplay,
)


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
if not torch.cuda.is_available():
    print("no GPU - go to Runtime > Change runtime type > GPU")


In [ ]:
os.makedirs("data", exist_ok=True)
os.makedirs("models/bert_sentiment_model", exist_ok=True)
os.makedirs("models/tokenizer", exist_ok=True)
os.makedirs("outputs", exist_ok=True)


## 2. Load data

IMDb dataset from Hugging Face's `datasets` lib. Full set is 25k/25k train/test, balanced. We just take a smaller balanced chunk of it.


In [ ]:
imdb = load_dataset("mteb/imdb")
imdb


In [ ]:
TRAIN_SIZE = 4000
VAL_SIZE = 1000
TEST_SIZE = 1000

def balanced_subset(split, n_total, seed=SEED):
    df = split.to_pandas()
    n = n_total // 2
    pos = df[df.label == 1].sample(n=n, random_state=seed)
    neg = df[df.label == 0].sample(n=n, random_state=seed)
    return pd.concat([pos, neg]).sample(frac=1, random_state=seed).reset_index(drop=True)

train_val_df = balanced_subset(imdb["train"], TRAIN_SIZE + VAL_SIZE)
test_df = balanced_subset(imdb["test"], TEST_SIZE)

train_df = train_val_df.iloc[:TRAIN_SIZE].reset_index(drop=True)
val_df = train_val_df.iloc[TRAIN_SIZE:].reset_index(drop=True)

print(train_df.shape, val_df.shape, test_df.shape)
train_df.head()


In [ ]:
pd.concat([
    train_df.assign(split="train"),
    val_df.assign(split="val"),
    test_df.assign(split="test"),
]).to_csv("data/imdb_dataset.csv", index=False)


In [ ]:
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(name, df.label.value_counts().to_dict())


## 3. A quick look at the data

In [ ]:
train_df.isnull().sum()


In [ ]:
train_df.label.map({0: "Negative", 1: "Positive"}).value_counts().plot(
    kind="bar", color=["#d62728", "#2ca02c"], figsize=(5, 4), title="Label distribution"
)
plt.tight_layout()
plt.savefig("outputs/label_distribution.png")
plt.show()


In [ ]:
train_df["word_count"] = train_df["text"].apply(lambda x: len(x.split()))
train_df["word_count"].describe()


In [ ]:
train_df["word_count"].hist(bins=40, figsize=(6, 4), color="#1f77b4")
plt.title("Review length (word count)")
plt.xlabel("words")
plt.tight_layout()
plt.savefig("outputs/review_length_histogram.png")
plt.show()


In [ ]:
from collections import Counter
import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

def words_only(text):
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    return [w for w in text.lower().split() if w not in stop_words and len(w) > 2]

all_words = []
for t in train_df["text"].sample(min(1000, len(train_df)), random_state=SEED):
    all_words.extend(words_only(t))

word_freq = Counter(all_words)
word_freq.most_common(20)


In [ ]:
from wordcloud import WordCloud

wc = WordCloud(width=800, height=400, background_color="white").generate_from_frequencies(word_freq)
plt.figure(figsize=(10, 5))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.tight_layout()
plt.savefig("outputs/wordcloud.png")
plt.show()


## 4. Cleaning




In [ ]:
def clean_text(text):
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

for df in (train_df, val_df, test_df):
    df["clean_text"] = df["text"].apply(clean_text)

train_df["clean_text"].iloc[0][:300]


## 5. Tokenization

In [ ]:
MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 256  # most reviews fit fine, and it's a lot faster than 512

tokenizer = BertTokenizerFast.from_pretrained(MODEL_NAME)

enc = tokenizer(train_df["clean_text"].iloc[0], truncation=True, max_length=MAX_LENGTH, padding="max_length")
tokenizer.decode(enc["input_ids"][:20])


In [ ]:
class IMDbDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=MAX_LENGTH):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
        )
        item = {k: torch.tensor(v) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_dataset = IMDbDataset(train_df["clean_text"], train_df["label"], tokenizer)
val_dataset = IMDbDataset(val_df["clean_text"], val_df["label"], tokenizer)
test_dataset = IMDbDataset(test_df["clean_text"], test_df["label"], tokenizer)

len(train_dataset), len(val_dataset), len(test_dataset)


# Traning






In [ ]:
model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)


In [ ]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_metric.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1_metric.compute(predictions=preds, references=labels)["f1"],
    }


In [ ]:
training_args = TrainingArguments(
    output_dir="outputs/checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=25,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer),
)


In [ ]:
trainer.train()


In [ ]:
log_history = trainer.state.log_history
train_loss = [(x["step"], x["loss"]) for x in log_history if "loss" in x]
eval_loss = [(x["step"], x["eval_loss"]) for x in log_history if "eval_loss" in x]

plt.figure(figsize=(7, 4))
if train_loss:
    s, l = zip(*train_loss)
    plt.plot(s, l, label="train loss")
if eval_loss:
    s, l = zip(*eval_loss)
    plt.plot(s, l, label="val loss", marker="o")
plt.xlabel("step")
plt.ylabel("loss")
plt.legend()
plt.tight_layout()
plt.savefig("outputs/training_loss.png")
plt.show()


## 7. Evaluation

In [ ]:
test_output = trainer.predict(test_dataset)
test_logits = test_output.predictions
test_labels = test_output.label_ids
test_probs = torch.softmax(torch.tensor(test_logits), dim=-1).numpy()
test_preds = np.argmax(test_logits, axis=-1)


In [ ]:
acc = accuracy_score(test_labels, test_preds)
precision, recall, f1, _ = precision_recall_fscore_support(test_labels, test_preds, average="binary")
try:
    auc = roc_auc_score(test_labels, test_probs[:, 1])
except Exception:
    auc = float("nan")

report = f"Accuracy : {acc:.4f}\nPrecision: {precision:.4f}\nRecall   : {recall:.4f}\nF1-score : {f1:.4f}\nROC-AUC  : {auc:.4f}"
print(report)

with open("outputs/evaluation_report.txt", "w") as f:
    f.write(report)


In [ ]:
cm = confusion_matrix(test_labels, test_preds)
ConfusionMatrixDisplay(cm, display_labels=["Negative", "Positive"]).plot(cmap="Blues", colorbar=False)
plt.title("Confusion matrix")
plt.tight_layout()
plt.savefig("outputs/confusion_matrix.png")
plt.show()


## 8. Try it on your own text

In [ ]:
model.eval()

def predict_sentiment(text, max_length=MAX_LENGTH):
    clean = clean_text(text)
    enc = tokenizer(clean, truncation=True, max_length=max_length, padding="max_length", return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**enc).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]
    label = "Positive" if probs[1] > probs[0] else "Negative"
    return label, float(max(probs))

sample_reviews = [
    "This movie was an absolute masterpiece, the acting and story blew me away!",
    "Waste of two hours. Poor script, worse acting, and a pointless plot.",
    "It was okay, some good moments but overall pretty forgettable.",
]

for r in sample_reviews:
    label, conf = predict_sentiment(r)
    print(f"{label} ({conf:.1%}) - {r}")


In [ ]:
review = input("Enter a review: ")
label, conf = predict_sentiment(review)
print(f"{label} ({conf:.1%})")


## 9. Save the model

In [ ]:
MODEL_DIR = "models/bert_sentiment_model"
TOKENIZER_DIR = "models/tokenizer"

model.save_pretrained(MODEL_DIR)
tokenizer.save_pretrained(TOKENIZER_DIR)
print("saved")


In [ ]:
!zip -r -q models.zip models
!zip -r -q outputs.zip outputs
# from google.colab import files
# files.download('models.zip')


## 10. Load it back

In [ ]:
loaded_tokenizer = BertTokenizerFast.from_pretrained(TOKENIZER_DIR)
loaded_model = BertForSequenceClassification.from_pretrained(MODEL_DIR)
loaded_model.to(device)
loaded_model.eval()

def predict_loaded(text, max_length=MAX_LENGTH):
    clean = clean_text(text)
    enc = loaded_tokenizer(clean, truncation=True, max_length=max_length, padding="max_length", return_tensors="pt").to(device)
    with torch.no_grad():
        logits = loaded_model(**enc).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]
    label = "Positive" if probs[1] > probs[0] else "Negative"
    return label, float(max(probs))

for r in sample_reviews:
    label, conf = predict_loaded(r)
    print(f"{label} ({conf:.1%}) - {r}")


In [ ]:
from lime.lime_text import LimeTextExplainer

explainer = LimeTextExplainer(class_names=["Negative", "Positive"])

def lime_predict_proba(texts):
    out = []
    for t in texts:
        clean = clean_text(t)
        enc = tokenizer(clean, truncation=True, max_length=MAX_LENGTH, padding="max_length", return_tensors="pt").to(device)
        with torch.no_grad():
            logits = model(**enc).logits
            probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]
        out.append(probs)
    return np.array(out)

exp = explainer.explain_instance(sample_reviews[0], lime_predict_proba, num_features=10, num_samples=200)
exp.as_list()


In [ ]:
exp.show_in_notebook(text=True)
